### Problem 001: Unique Paths (LeetCode 62)

### Problem Definition and Constraints
There is an `m x n` grid where you are allowed to move either **down** or **to the right** at any point in time. 
Given the two integers `m` and `n`, return the number of possible unique paths that can be taken from the top-left corner of the grid (`grid[0][0]`) to the bottom-right corner (`grid[m - 1][n - 1]`).

**Examples:**
* **Example 1:**
  * **Input:** `m = 3, n = 6`
  * **Output:** `28`
* **Example 2:**
  * **Input:** `m = 3, n = 2`
  * **Output:** `3`

**Constraints:**
* 1 <= m, n <= 100

### Core Logic: The "Where Did I Come From?" Principle
Instead of trying to trace every possible path from start to finish (which branches exponentially), we flip the question. 

Imagine you are standing in a random room in the middle of this grid. Because you are only allowed to move **Down** or **Right**, there are only *two possible doors* you could have used to enter your current room:
1. The door from the room directly **Above** you.
2. The door from the room directly to your **Left**.

Therefore, the total number of unique ways to reach your current room is simply:
**(Ways to reach the room Above) + (Ways to reach the room to the Left)**

**The Base Cases (The Edges):**
What if you are in the very top row? There is no room above you. You could only have marched straight across from the left. There is only exactly **1** way to reach any room in the top row.
Similarly, for the leftmost column, there is no room to your left. You could only have dropped straight down. There is exactly **1** way to reach any room in the left column.

### Approach 1: 2D Dynamic Programming ($O(m \times n)$ Space)
We create a 2D matrix (a grid of rows and columns). We pre-fill the top row and left column with `1`s. Then, we loop through the remaining empty rooms one by one, adding the value from the room above and the room to the left. 
* **Time Complexity:** $O(m \times n)$ — We visit every room in the grid exactly once.
* **Space Complexity:** $O(m \times n)$ — We store the entire 2D grid in memory.

### Approach 2: Space Optimization ($O(n)$ Space)
If you look closely at the engine `dp[r][c] = dp[r-1][c] + dp[r][c-1]`, you realize that to calculate the current room, we *only* ever look at the current row we are on, and the single row directly above us. The rest of the historical grid above that is completely useless.
Instead of keeping a massive 2D grid, we can just keep a single 1D array that represents the "Row Above." As we calculate our new current row, we just overwrite the old values. 
* **Time Complexity:** $O(m \times n)$
* **Space Complexity:** $O(n)$ — We only store a single row (the width of the grid) in memory.

In [12]:
class Solution:
    # ---------------------------------------------------------
    # Approach 1: Standard 2D DP Matrix (O(m * n) Space)
    # ---------------------------------------------------------
    def uniquePaths_2D(self, m: int, n: int) -> int:
        # Create an m x n matrix filled with 1s. 
        # Why 1s? This brilliantly handles our base cases automatically!
        # The entire top row (r=0) and left column (c=0) will start with 1,
        # which is exactly what we want because there is only 1 way to reach them.
        dp = [[1] * n for _ in range(m)]
        
        # Start iterating from row 1 and col 1 (skipping the base case edges)
        for r in range(1, m):
            for c in range(1, n):
                # The Core Engine: 
                # Ways to reach here = (Ways to reach Above) + (Ways to reach Left)
                dp[r][c] = dp[r - 1][c] + dp[r][c - 1]
                
        # The bottom-right corner now holds the accumulation of all possible paths
        return dp[m - 1][n - 1]

    # ---------------------------------------------------------
    # Approach 2: 1D Sliding Row (O(n) Space)
    # ---------------------------------------------------------
    def uniquePaths(self, m: int, n: int) -> int:
        # We only keep track of a single row at a time.
        # It starts representing the very top row, so it's all 1s.
        row = [1] * n
        
        # Loop through the remaining rows (from row 1 down to row m-1)
        for r in range(1, m):
            # For each column, calculate the new path combinations
            # We can start at column 1, because the leftmost edge (col 0) is always 1
            for c in range(1, n):
                
                # The Core Engine (Optimized):
                # row[c] currently holds the value from the row ABOVE us.
                # row[c - 1] was just updated in the previous loop iteration, 
                # so it holds the fresh value from our LEFT.
                # We add them together and overwrite our current spot.
                row[c] = row[c] + row[c - 1]
                
        # Once we finish all rows, the very last item in our array is the bottom-right room.
        return row[n - 1]

### Problem 002: Longest Common Subsequence (LeetCode 1143)

### Problem Definition and Constraints
Given two strings `text1` and `text2`, return the length of the longest common subsequence between the two strings if one exists, otherwise return `0`.
A subsequence is a sequence that can be derived from the given sequence by deleting some or no elements without changing the relative order of the remaining characters.
A common subsequence of two strings is a subsequence that exists in both strings.

**Examples:**
* **Example 1:**
  * **Input:** `text1 = "cat", text2 = "crabt"`
  * **Output:** `3`
  * *Explanation:* The longest common subsequence is "cat" which has a length of 3.
* **Example 2:**
  * **Input:** `text1 = "abcd", text2 = "abcd"`
  * **Output:** `4`
* **Example 3:**
  * **Input:** `text1 = "abcd", text2 = "efgh"`
  * **Output:** `0`

**Constraints:**
* 1 <= text1.length, text2.length <= 1000
* `text1` and `text2` consist of only lowercase English characters.

### Core Logic: The "Grid of Decisions" (Diagonal vs. Lateral)
Imagine setting this up as a 2D grid, where `text1` forms the rows and `text2` forms the columns. We add an extra "empty" row at the top and an "empty" column on the left, initialized to `0`, representing comparing against an empty string.

As we evaluate each cell (comparing a character from `text1` against `text2`), we face two scenarios:
1. **The Match (Diagonal Move):** The characters are identical! We found a piece of our common subsequence. We gain `1` point. Because we used both characters, we must look at the best score we had *before* either of these characters were introduced. Physically, this means looking at the cell diagonally up and to the left, and adding `1`.
2. **The Mismatch (Lateral Move):** The characters don't match. We can't gain a point. We have to "delete" a character from one of the strings to keep searching. We look at two options:
   * Delete from `text1` (Look at the cell directly Above).
   * Delete from `text2` (Look at the cell directly to the Left).
   We simply take the maximum score of those two options and carry it forward.

### Approach 1: 2D Dynamic Programming ($O(m \times n)$ Space)
We build an `(m + 1) x (n + 1)` matrix initialized to `0`. We loop through every combination of characters. If they match, we pull from the diagonal + 1. If they don't, we take the max of the top or left cell.
* **Time Complexity:** $O(m \times n)$ — We evaluate every character pairing exactly once.
* **Space Complexity:** $O(m \times n)$ — We store the entire 2D grid.

### Approach 2: Space Optimization ($O(\min(m, n))$ Space)
Just like in *Unique Paths*, calculating the current row only requires data from the *current row* (Left) and the *previous row* (Above / Diagonal). We do not need the entire historical matrix. We can optimize this by only keeping two 1D arrays: one for the previous row and one for the current row we are building.
* **Time Complexity:** $O(m \times n)$
* **Space Complexity:** $O(\min(m, n))$ — We ensure the 1D arrays are sized to match the shorter string to save maximum space.

In [13]:
class Solution:
    # ---------------------------------------------------------
    # Approach 1: Standard 2D DP Matrix (O(m * n) Space)
    # ---------------------------------------------------------
    def longestCommonSubsequence_2D(self, text1: str, text2: str) -> int:
        m, n = len(text1), len(text2)
        
        # Create an (m+1) x (n+1) grid filled with 0s.
        # The extra row and column handle the base case: 
        # an empty string shares 0 common characters with any other string.
        dp = [[0] * (n + 1) for _ in range(m + 1)]
        
        # Loop through every character in text1 (rows)
        for r in range(1, m + 1):
            # Loop through every character in text2 (cols)
            for c in range(1, n + 1):
                
                # Note: r-1 and c-1 because our strings are 0-indexed, 
                # but our DP array is 1-indexed (shifted by 1 for the empty bases)
                if text1[r - 1] == text2[c - 1]:
                    # Match! Take the diagonal (score before both chars) + 1
                    dp[r][c] = 1 + dp[r - 1][c - 1]
                else:
                    # Mismatch! Take the max of ignoring text1's char (Above) 
                    # OR ignoring text2's char (Left)
                    dp[r][c] = max(dp[r - 1][c], dp[r][c - 1])
                    
        # The bottom-right corner holds the length of the complete LCS
        return dp[m][n]

    # ---------------------------------------------------------
    # Approach 2: 2-Row Space Optimization (O(min(m, n)) Space)
    # ---------------------------------------------------------
    def longestCommonSubsequence(self, text1: str, text2: str) -> int:
        # Optimization: Make text2 the shorter string to minimize our array size
        if len(text1) < len(text2):
            text1, text2 = text2, text1
            
        m, n = len(text1), len(text2)
        
        # We only need two rows: the one above us, and the one we are building
        prev_row = [0] * (n + 1)
        curr_row = [0] * (n + 1)
        
        for r in range(1, m + 1):
            for c in range(1, n + 1):
                if text1[r - 1] == text2[c - 1]:
                    # Diagonal value is in prev_row at the previous column
                    curr_row[c] = 1 + prev_row[c - 1]
                else:
                    # Max of Above (prev_row[c]) and Left (curr_row[c-1])
                    curr_row[c] = max(prev_row[c], curr_row[c - 1])
                    
            # The current row is finished. It now becomes the "previous row" 
            # for the next iteration.
            # We must copy the values (or reassign references carefully)
            prev_row = curr_row[:]
            
        # The final answer lives at the end of the last evaluated row
        return prev_row[n]

### Problem 003: Best Time to Buy and Sell Stock with Cooldown (LeetCode 309)

### Problem Definition and Constraints
You are given an integer array `prices` where `prices[i]` is the price of a stock on the `i`th day.
You may buy and sell one stock multiple times with the following restrictions:
*   After you sell your stock, you cannot buy another one on the next day (i.e., there is a cooldown period of one day).
*   You may only own at most one stock at a time.
*   You may complete as many transactions as you like.
Return the maximum profit you can achieve.

**Examples:**
* **Example 1:**
  * **Input:** `prices = [1,3,4,0,4]`
  * **Output:** `6`
  * *Explanation:* Buy Day 0 (1), Sell Day 1 (3), Profit = 2. Cooldown Day 2. Buy Day 3 (0), Sell Day 4 (4), Profit = 4. Total = 6.
* **Example 2:**
  * **Input:** `prices = [1]`
  * **Output:** `0`

**Constraints:**
* 1 <= prices.length <= 5000
* 0 <= prices[i] <= 1000

### Core Logic: The State Machine (The Three Hotel Rooms)
Instead of a standard 2D grid, this problem introduces a **State Machine**. Imagine you are a trader walking through a calendar day by day. Every night, based on your actions, you must sleep in one of three hotel rooms. You want to track the maximum money you could possibly have in your wallet while sleeping in each room.

1.  **The "Holding" Room (You own a stock):**
    *   *How you got here:* You either slept in this room yesterday and did nothing, OR you were in the "Resting" room yesterday and bought a stock today (subtracting the stock price from your wallet).
2.  **The "Sold" Room (You just sold today):**
    *   *How you got here:* You *must* have been in the "Holding" room yesterday, and you sold your stock today (adding the stock price to your wallet). You are forced to leave this room tomorrow.
3.  **The "Resting" Room (Empty hands, ready to buy):**
    *   *How you got here:* You either slept in this room yesterday and did nothing, OR you slept in the "Sold" room yesterday (serving your mandatory 1-day cooldown) and transitioned here today.

At the end of the timeline, your maximum profit will be the most money in either the "Resting" room or the "Sold" room (you would never end the timeline in the "Holding" room, because you could have just sold it for extra cash).

### Approach 1: Parallel DP Arrays (O(n) Space)
We create three arrays: `hold`, `sold`, and `rest`, where `hold[i]` represents the max profit on day `i` if we end the day in the Holding state. We calculate day `i` by looking at the states from day `i-1`.
* **Time Complexity:** O(n) — We iterate through the prices array exactly once.
* **Space Complexity:** O(n) — We maintain three arrays of size n.

### Approach 2: Sliding Variables (O(1) Space)
Just like in *Unique Paths* and *Longest Common Subsequence*, to calculate today's hotel rooms, we only need to look at *yesterday's* hotel rooms. The rest of the history is useless. We can replace the arrays with three simple variables that update as we walk through the days.
* **Time Complexity:** O(n)
* **Space Complexity:** O(1) — We only track three variables representing our current state.

In [14]:
from typing import List

class Solution:
    # ---------------------------------------------------------
    # Approach 1: Parallel DP Arrays (O(n) Space)
    # ---------------------------------------------------------
    def maxProfit_array(self, prices: List[int]) -> int:
        if not prices:
            return 0
            
        n = len(prices)
        hold = [0] * n  # Ledger tracking max money if we end the day holding a stock
        sold = [0] * n  # Ledger tracking max money if we end the day having just sold
        rest = [0] * n  # Ledger tracking max money if we end the day doing nothing (cooldown/waiting)
        
        # Base Cases (Day 0)
        hold[0] = -prices[0] # We bought the stock on Day 0, wallet becomes negative to reflect the purchase cost.
        sold[0] = float('-inf') # Impossible to sell on Day 0 because we didn't own a stock before today.
        rest[0] = 0 # Doing nothing on Day 0 leaves our wallet at the starting balance of $0.
        
        for i in range(1, n):
            # To be Holding today: keep the cheapest historical buy (hold[i-1]), 
            # OR use our banked cash (rest[i-1]) to buy at today's price.
            hold[i] = max(hold[i - 1], rest[i - 1] - prices[i])
            
            # To be Sold today: MUST have held a stock yesterday. We sell it at today's price to lock in profit.
            sold[i] = hold[i - 1] + prices[i]
            
            # To be Resting today: keep resting with our current banked cash (rest[i-1]), 
            # OR absorb the money from yesterday's sale (sold[i-1]), serving our 1-day cooldown.
            rest[i] = max(rest[i - 1], sold[i - 1])
            
        # We want the max money without holding a stock at the very end (holding at the end is a waste).
        return max(rest[n - 1], sold[n - 1])

    # ---------------------------------------------------------
    # Approach 2: Sliding Variables (O(1) Space)
    # ---------------------------------------------------------
    def maxProfit(self, prices: List[int]) -> int:
        if not prices:
            return 0
            
        # The Memory (Base Cases for before Day 0)
        # We start with $0. It is impossible to hold or have just sold before the market opens.
        hold = float('-inf') 
        sold = float('-inf')
        rest = 0 
        
        for price in prices:
            # We must remember yesterday's 'sold' value before we overwrite it with today's math,
            # because today's 'rest' calculation requires yesterday's 'sold' value (the cooldown delay).
            prev_sold = sold
            
            # The State Transitions (Moving between the hotel rooms)
            
            # 1. Update Sold: Take the cheapest buy we are holding, and sell it at today's price.
            sold = hold + price
            
            # 2. Update Hold: Keep holding our previous best buy, OR use our resting bank account to buy today's stock.
            hold = max(hold, rest - price)
            
            # 3. Update Rest: Keep our current resting bank account, OR absorb yesterday's sale profit into the bank.
            rest = max(rest, prev_sold)
            
        # Return the max wallet balance from the states where we don't own a stock.
        return max(rest, sold)

### Problem 004: Coin Change II (LeetCode 518)

### Problem Definition and Constraints
You are given an integer array `coins` representing coins of different denominations and an integer `amount` representing a target amount of money.
Return the **number of distinct combinations** that total up to `amount`. If it's impossible to make up the amount, return `0`.
You have an unlimited number of each coin.

**Examples:**
* **Example 1:**
  * **Input:** `amount = 4, coins = [1,2,3]`
  * **Output:** `4`
  * **Explanation:** `1+1+1+1`, `1+1+2`, `2+2`, `1+3`.
* **Example 2:**
  * **Input:** `amount = 7, coins = [2,4]`
  * **Output:** `0`

* Constraints:
  * 1 <= coins.length <= 100
  * 1 <= coins[i] <= 5000
  * 0 <= amount <= 5000

### Core Logic: Combinations vs. Permutations (The Outer Loop Trap)
If you solve this using the standard 1D DP from "Climbing Stairs" (where the outer loop is the `amount` and the inner loop is the `coins`), you will accidentally calculate **permutations**. It will count `1 + 3` and `3 + 1` as two completely different ways to make 4, which violates the requirement for distinct combinations.

To enforce **combinations**, we must swap the loops. 
*   **The Outer Loop:** We iterate through the `coins`. 
*   **The Inner Loop:** We iterate through the `amounts`.

By putting the coins on the outside, we mathematically force the chronological order. We calculate all possible ways to make amounts using *strictly* the `1` coin. Then, we sweep through again, layering on the combinations that use the `2` coin. This guarantees that `1 + 3` is evaluated, but `3 + 1` is completely impossible to generate, naturally eliminating duplicates.

### Approach: 1D Dynamic Programming (Bottom-Up)
1. Initialize a `dp` array of size `amount + 1` filled with `0`s. 
2. **Base Case:** `dp[0] = 1`. There is exactly 1 way to make an amount of 0: use zero coins.
3. **The Coin Sweep:** Iterate through every `coin` in `coins`.
4. **The Amount Sweep:** For the current `coin`, iterate through every amount `a` starting from `coin` up to `amount`. (We start at `coin` because it's impossible to use a $5 coin to make an amount of $3).
5. **The Accumulation:** The number of ways to make amount `a` increases by the number of ways we already figured out how to make `a - coin`. `dp[a] += dp[a - coin]`.
6. Return the final value at `dp[amount]`.

* **Time Complexity:** $O(n \cdot a)$ — Where $n$ is the number of coins and $a$ is the target amount. We evaluate every amount for every coin exactly once.
* **Space Complexity:** $O(a)$ — We compress the 2D matrix into a single 1D array of size `amount + 1`.

In [15]:
from typing import List

class Solution:
    def change(self, amount: int, coins: List[int]) -> int:
        
        # dp[a] represents the number of distinct combinations to make amount 'a'
        dp = [0] * (amount + 1)
        
        # Base case: There is exactly 1 way to make an amount of 0 (use no coins)
        dp[0] = 1
        
        # The Outer Loop MUST be the coins to ensure we only generate distinct 
        # combinations and avoid counting permutations like (1,3) and (3,1).
        for coin in coins:
            
            # The Inner Loop: evaluate every amount that is large enough to use this coin
            for a in range(coin, amount + 1):
                
                # The number of ways to make amount 'a' increases by the number of ways
                # we already know how to make the remaining balance ('a - coin').
                dp[a] += dp[a - coin]
                
        # The final target amount holds the accumulation of all valid paths
        return dp[amount]


        """
        =========================================================
        SIMULATION NOTES: amount = 4, coins = [1, 2, 3]
        =========================================================
        Initial State: dp = [1, 0, 0, 0, 0] (Indices 0 through 4)
        
        Coin = 1: (We can only use 1s)
          a = 1: dp[1] += dp[0] (1) -> dp = [1, 1, 0, 0, 0]
          a = 2: dp[2] += dp[1] (1) -> dp = [1, 1, 1, 0, 0]
          a = 3: dp[3] += dp[2] (1) -> dp = [1, 1, 1, 1, 0]
          a = 4: dp[4] += dp[3] (1) -> dp = [1, 1, 1, 1, 1]
          *There is exactly 1 way to make any amount using only 1s.*
          
        Coin = 2: (We can now use 1s and 2s)
          a = 2: dp[2] += dp[0] (1) -> dp = [1, 1, 2, 1, 1]  (1+1, 2)
          a = 3: dp[3] += dp[1] (1) -> dp = [1, 1, 2, 2, 1]  (1+1+1, 1+2)
          a = 4: dp[4] += dp[2] (2) -> dp = [1, 1, 2, 2, 3]  (1+1+1+1, 1+1+2, 2+2)
          
        Coin = 3: (We can use 1s, 2s, and 3s)
          a = 3: dp[3] += dp[0] (1) -> dp = [1, 1, 2, 3, 3]
          a = 4: dp[4] += dp[1] (1) -> dp = [1, 1, 2, 3, 4]  (Adds the 1+3 combo)
          
        Final Answer: dp[4] = 4.
        """

### Problem 005: Target Sum (LeetCode 494)

### Problem Definition and Constraints
You are given an integer array `nums` and an integer `target`.
For each number in the array, you must choose to assign a `+` (add) or a `-` (subtract) to it. 
Return the total number of different valid mathematical expressions you can build that evaluate to exactly `target`.

**Examples:**
* **Example 1:**
  * **Input:** `nums = [2,2,2], target = 2`
  * **Output:** `3`
  * **Explanation:** There are 3 ways to reach 2:
    `+2 +2 -2 = 2`
    `+2 -2 +2 = 2`
    `-2 +2 +2 = 2`

* Constraints:
  * 1 <= nums.length <= 20
  * 0 <= nums[i] <= 1000
  * -1000 <= target <= 1000

### Core Logic: Level-by-Level State Tracking (Hash Map DP)
If we use standard recursion, the decision tree branches by 2 at every step (add or subtract). For an array of 20 elements, this is $2^{20}$ (over 1 million) branches. 

To optimize this, we recognize a massive amount of overlapping subproblems. If our target is 5, and halfway through the array we reach a current sum of 2, it does not matter *how* we reached 2. We only care that we are currently at 2, and we have a specific set of numbers remaining.

Instead of a bulky 2D array with shifted indices to handle negative sums, we can use a **Hash Map** to compress our state. 
*   The Hash Map tracks `{current_sum : number_of_ways_to_reach_it}`.
*   We start with an empty state: `{0 : 1}` (There is exactly 1 way to reach a sum of 0 using 0 numbers).
*   For every number in `nums`, we create a brand new map. We look at every `(current_sum, ways)` in our previous map, and branch it into two new possibilities in our new map: 
    *   `new_map[current_sum + num] += ways`
    *   `new_map[current_sum - num] += ways`
*   After processing all numbers, we simply look up the `target` key in our final map.

### Approach: Bottom-Up Dictionary
1. Initialize a dictionary `dp` with `{0: 1}`.
2. Iterate through every `n` in `nums`.
3. Inside the loop, initialize a temporary `next_dp` dictionary.
4. Iterate through `val` (the sum) and `count` (the ways to reach it) in `dp.items()`.
5. Add `count` to `next_dp[val + n]`.
6. Add `count` to `next_dp[val - n]`.
7. Reassign `dp = next_dp`.
8. Once the array is fully processed, return `dp.get(target, 0)`.

* **Time Complexity:** $O(N \cdot M)$ — Where $N$ is the length of `nums` and $M$ is the sum of all elements in the array. The hash map size is strictly bounded by the maximum possible positive and negative sum.
* **Space Complexity:** $O(M)$ — The hash map will hold at most $2M + 1$ distinct sums at any given level.

In [16]:
import collections
from typing import List

class Solution:
    def findTargetSumWays(self, nums: List[int], target: int) -> int:
        
        # dp tracks { current_sum : number_of_ways_to_reach_this_sum }
        # Base case: 1 way to reach sum 0 with no elements used.
        dp = collections.defaultdict(int)
        dp[0] = 1
        
        for n in nums:
            # Create a new map for the next level of the decision tree
            next_dp = collections.defaultdict(int)
            
            for current_sum, count in dp.items():
                
                # Choice 1: Add the number
                next_dp[current_sum + n] += count
                
                # Choice 2: Subtract the number
                next_dp[current_sum - n] += count
                
            # Slide the window forward: the new map becomes our starting point for the next number
            dp = next_dp
            
        # Return the number of ways to reach the exact target sum. 
        # If the target is impossible, get() safely returns 0.
        return dp.get(target, 0)


        """
        =========================================================
        SIMULATION NOTES: nums = [2, 2, 2], target = 2
        =========================================================
        Initial State: dp = {0: 1}
        
        Loop 1: n = 2
          current_sum = 0, count = 1
          next_dp[0 + 2] += 1  -> next_dp[2] = 1
          next_dp[0 - 2] += 1  -> next_dp[-2] = 1
          dp becomes: {2: 1, -2: 1}
          
        Loop 2: n = 2
          current_sum = 2, count = 1
            next_dp[2 + 2] += 1  -> next_dp[4] = 1
            next_dp[2 - 2] += 1  -> next_dp[0] += 1
          current_sum = -2, count = 1
            next_dp[-2 + 2] += 1 -> next_dp[0] += 1 (Now it is 2!)
            next_dp[-2 - 2] += 1 -> next_dp[-4] = 1
          dp becomes: {4: 1, 0: 2, -4: 1}
          
        Loop 3: n = 2
          current_sum = 4, count = 1
            next_dp[4 + 2] += 1  -> next_dp[6] = 1
            next_dp[4 - 2] += 1  -> next_dp[2] += 1
          current_sum = 0, count = 2
            next_dp[0 + 2] += 2  -> next_dp[2] += 2 (Now it is 3!)
            next_dp[0 - 2] += 2  -> next_dp[-2] += 2
          current_sum = -4, count = 1
            next_dp[-4 + 2] += 1 -> next_dp[-2] += 1 (Now it is 3!)
            next_dp[-4 - 2] += 1 -> next_dp[-6] = 1
          dp becomes: {6: 1, 2: 3, -2: 3, -6: 1}
          
        Final Answer: dp.get(2) -> returns 3.
        """

### Problem 006: Interleaving String (LeetCode 97)

### Problem Definition and Constraints
You are given three strings `s1`, `s2`, and `s3`. Return `true` if `s3` is formed by interleaving `s1` and `s2` together, or `false` otherwise.
Interleaving divides `s1` and `s2` into substrings and alternates them to form `s3` while perfectly maintaining the relative order of the characters from both original strings.

**Examples:**
* **Example 1:**
  * **Input:** `s1 = "aaaa", s2 = "bbbb", s3 = "aabbbbaa"`
  * **Output:** `true`
  * **Explanation:** `s1` is split into `["aa", "aa"]`, `s2` remains `"bbbb"`. They interleave as `"aa" + "bbbb" + "aa"`.
* **Example 2:**
  * **Input:** `s1 = "abc", s2 = "xyz", s3 = "abxzcy"`
  * **Output:** `false`
  * **Explanation:** You cannot form `s3` without breaking the chronological order of the letters in `s1` or `s2`.

* Constraints:
  * 0 <= s1.length, s2.length <= 100
  * 0 <= s3.length <= 200
  * Consists of lowercase English letters.

### Core Logic: The 2D Decision Grid
The very first check is a simple math filter: if `len(s1) + len(s2) != len(s3)`, it is physically impossible to interleave them, so return `False`.

If lengths match, this problem becomes a 2D grid pathfinding problem, heavily mirroring **Unique Paths**.
Imagine a grid where `s1` forms the rows and `s2` forms the columns. You start at the top-left `(0, 0)` and want to reach the bottom-right `(len(s1), len(s2))`.
At any cell `(i, j)`, you are trying to match the character in `s3` at index `i + j`. You have a maximum of two choices:
1. **Move Down (Use `s1`):** If `s1[i] == s3[i+j]`, you can move down to `(i+1, j)`.
2. **Move Right (Use `s2`):** If `s2[j] == s3[i+j]`, you can move right to `(i, j+1)`.

If we evaluate this grid from bottom-to-top, right-to-left (Bottom-Up DP), we can cache which paths successfully reach the end.

### Approach: 2D Dynamic Programming (Bottom-Up)
1. Perform the length check: `len(s1) + len(s2) == len(s3)`.
2. Initialize a 2D boolean `dp` matrix of size `(len(s1) + 1) x (len(s2) + 1)` filled with `False`.
3. Set the base case: `dp[len(s1)][len(s2)] = True`. (If you have reached the end of both strings, you have successfully formed `s3`).
4. Iterate `i` from `len(s1)` down to 0, and `j` from `len(s2)` down to 0.
5. For each cell `(i, j)`, evaluate the two choices:
   * Is `i < len(s1)` AND `s1[i] == s3[i + j]` AND the cell below (`dp[i+1][j]`) is `True`? If so, `dp[i][j] = True`.
   * Is `j < len(s2)` AND `s2[j] == s3[i + j]` AND the cell to the right (`dp[i][j+1]`) is `True`? If so, `dp[i][j] = True`.
6. Return `dp[0][0]`.

* **Time Complexity:** $O(m \cdot n)$ — Where $m$ is the length of `s1` and $n$ is the length of `s2`. We evaluate every combination of indices exactly once.
* **Space Complexity:** $O(m \cdot n)$ — To store the 2D boolean matrix. (Note: Like Unique Paths, this can be mathematically compressed to $O(n)$ space by only keeping track of a single sliding row, but the 2D matrix is standard for interviews).

In [17]:
class Solution:
    def isInterleave(self, s1: str, s2: str, s3: str) -> bool:
        # 1. The Math Filter
        if len(s1) + len(s2) != len(s3):
            return False
            
        m, n = len(s1), len(s2)
        
        # 2. Initialize the DP grid with False
        dp = [[False] * (n + 1) for _ in range(m + 1)]
        
        # 3. Base Case: The very bottom-right corner is True
        dp[m][n] = True
        
        # 4. Sweep backward through the grid
        for i in range(m, -1, -1):
            for j in range(n, -1, -1):
                
                # Choice 1: Can we consume a character from s1?
                # We check bounds, match the character against s3, 
                # and verify if the subsequent path (Below) is valid.
                if i < m and s1[i] == s3[i + j] and dp[i + 1][j]:
                    dp[i][j] = True
                    
                # Choice 2: Can we consume a character from s2?
                # We check bounds, match the character against s3,
                # and verify if the subsequent path (Right) is valid.
                if j < n and s2[j] == s3[i + j] and dp[i][j + 1]:
                    dp[i][j] = True
                    
        # 5. The top-left corner holds the result for the entire string
        return dp[0][0]


        """
        =========================================================
        SIMULATION NOTES: s1 = "a", s2 = "b", s3 = "ab"
        =========================================================
        m = 1, n = 1. Grid size is 2x2.
        Initial dp:
        [F, F]
        [F, T] (Base case set at dp[1][1])
        
        i = 1, j = 0: (Bottom row, checking s2 against s3)
          j < 1 -> 0 < 1 (True)
          s2[0] == s3[1+0] -> 'b' == 'b' (True)
          dp[1][1] is True.
          dp[1][0] becomes True.
          
        i = 0, j = 1: (Right col, checking s1 against s3)
          i < 1 -> 0 < 1 (True)
          s1[0] == s3[0+1] -> 'a' == 'b' (False)
          dp[0][1] remains False.
          
        i = 0, j = 0: (Top-left, checking both)
          Choice 1 (s1): 
            0 < 1 (True)
            s1[0] == s3[0] -> 'a' == 'a' (True)
            dp[1][0] (Below) is True.
            dp[0][0] becomes True!
            
        Final dp grid:
        [T, F]
        [T, T]
        
        Returns dp[0][0] -> True.
        """

### Problem 007: Edit Distance (LeetCode 72)

### Problem Definition and Constraints
You are given two strings `word1` and `word2`.
You are allowed to perform three operations on `word1` an unlimited number of times:
*   **Insert** a character
*   **Delete** a character
*   **Replace** a character

Return the minimum number of operations required to convert `word1` into `word2`.

**Examples:**
* **Example 1:**
  * **Input:** `word1 = "monkeys", word2 = "money"`
  * **Output:** `2`
  * **Explanation:**
    monkeys -> monkey (remove 's')
    monkey -> money (remove 'k')
* **Example 2:**
  * **Input:** `word1 = "neatcdee", word2 = "neetcode"`
  * **Output:** `3`
  * **Explanation:**
    neatcdee -> neetcdee (replace 'a' with 'e')
    neetcdee -> neetcde (remove last 'e')
    neetcde -> neetcode (insert 'o')

* Constraints:
  * 0 <= word1.length, word2.length <= 100
  * Both consist of lowercase English letters.

### Core Logic: The 3-Way Branching Grid
This is the pinnacle of 2D Dynamic Programming string alignment. We evaluate the strings character by character using a 2D grid where `word1` forms the rows and `word2` forms the columns.

When comparing `word1[i]` against `word2[j]`, we have two main realities:
1. **The Match:** `word1[i] == word2[j]`. The characters are already aligned. It costs `0` operations! We simply move diagonally to `dp[i+1][j+1]` to check the rest of the string.
2. **The Mismatch:** The characters are different. We must spend exactly `1` operation, but we get to choose the cheapest of three distinct paths:
   *   **Insert:** We pretend we inserted `word2[j]` into `word1`. This successfully matches the current character in `word2`, so we advance `j` (move Right), but we stay on `i` because we haven't actually consumed `word1[i]` yet. -> `dp[i][j+1]`
   *   **Delete:** We throw away `word1[i]`. We advance `i` (move Down), but stay on `j` to try matching `word2[j]` against the next character. -> `dp[i+1][j]`
   *   **Replace:** We overwrite `word1[i]` with `word2[j]`. Both characters are now successfully matched, so we advance both (move Diagonally). -> `dp[i+1][j+1]`

**The Base Cases (The Edges):**
What happens if we reach the end of `word1`, but `word2` still has 3 characters left? We are forced to use 3 Insert operations.
What if we reach the end of `word2`, but `word1` has 4 characters left? We are forced to use 4 Delete operations.
We initialize the bottom row and rightmost column of our grid to reflect these exact remaining counts.

### Approach: 2D Dynamic Programming (Bottom-Up)
1. Initialize a 2D `dp` array of size `(len(word1) + 1) x (len(word2) + 1)`.
2. Pre-fill the base cases:
   * Fill the bottom row so that `dp[len(word1)][j]` equals the remaining characters in `word2` (`len(word2) - j`).
   * Fill the rightmost column so that `dp[i][len(word2)]` equals the remaining characters in `word1` (`len(word1) - i`).
3. Loop `i` backward from `len(word1) - 1` down to 0, and `j` backward from `len(word2) - 1` down to 0.
4. Apply the core engine:
   * If characters match: `dp[i][j] = dp[i + 1][j + 1]`
   * If mismatch: `dp[i][j] = 1 + min(dp[i + 1][j], dp[i][j + 1], dp[i + 1][j + 1])`
5. Return the top-left corner `dp[0][0]`.

* **Time Complexity:** $O(m \cdot n)$ — Where $m$ is the length of `word1` and $n$ is the length of `word2`. We evaluate every cell in the grid exactly once.
* **Space Complexity:** $O(m \cdot n)$ — To store the 2D matrix. (Can be mathematically optimized to $O(\min(m, n))$ space by keeping only the current and previous rows, but the 2D matrix is vastly easier to implement).

In [18]:
class Solution:
    def minDistance(self, word1: str, word2: str) -> int:
        m, n = len(word1), len(word2)
        
        # Initialize an (m+1) x (n+1) grid filled with infinity
        dp = [[float('inf')] * (n + 1) for _ in range(m + 1)]
        
        # Base Case 1: Bottom Row
        # If word1 is empty, we must INSERT all remaining characters of word2.
        for j in range(n + 1):
            dp[m][j] = n - j
            
        # Base Case 2: Right Column
        # If word2 is empty, we must DELETE all remaining characters of word1.
        for i in range(m + 1):
            dp[i][n] = m - i
            
        # Bottom-Up Sweep
        for i in range(m - 1, -1, -1):
            for j in range(n - 1, -1, -1):
                
                # The Match: No operation required, inherit the diagonal score.
                if word1[i] == word2[j]:
                    dp[i][j] = dp[i + 1][j + 1]
                    
                # The Mismatch: Take 1 operation + the absolute best of the 3 branching paths.
                else:
                    dp[i][j] = 1 + min(
                        dp[i + 1][j],      # Delete (Move Down)
                        dp[i][j + 1],      # Insert (Move Right)
                        dp[i + 1][j + 1]   # Replace (Move Diagonally)
                    )
                    
        # The top-left corner holds the answer for the fully evaluated strings
        return dp[0][0]


        """
        =========================================================
        SIMULATION NOTES: word1 = "a", word2 = "b"
        =========================================================
        m = 1, n = 1. Grid size is 2x2.
        
        Base Cases Applied:
        dp[1][0] = 1, dp[1][1] = 0 (Bottom Row)
        dp[0][1] = 1, dp[1][1] = 0 (Right Col)
        
        Grid currently looks like this:
        [inf, 1]
        [  1, 0]
        
        Evaluate cell (0, 0): 'a' vs 'b'
        Mismatch! 
        Insert = dp[0][1] -> 1
        Delete = dp[1][0] -> 1
        Replace = dp[1][1] -> 0
        
        dp[0][0] = 1 + min(1, 1, 0) = 1 + 0 = 1.
        
        Final Grid:
        [1, 1]
        [1, 0]
        
        Returns dp[0][0] -> 1 operation (Replace 'a' with 'b').
        """

### Problem 008: Longest Increasing Path in a Matrix (LeetCode 329) [HARD]

### Problem Definition and Constraints
You are given an `m x n` integer matrix. Return the length of the longest strictly increasing path within the matrix.
From each cell, you can move to four directions: left, right, up, or down. You may not move diagonally or move outside the boundary.

**Examples:**
* **Example 1:**
  * **Input:** `matrix = [[5,5,3],[2,3,6],[1,1,1]]`
  * **Output:** `4`
  * **Explanation:** The longest increasing path is `[1, 2, 3, 6]` or `[1, 2, 3, 5]`.
* **Example 2:**
  * **Input:** `matrix = [[1,2,3],[2,1,4],[7,6,5]]`
  * **Output:** `7`
  * **Explanation:** The longest increasing path is `[1, 2, 3, 4, 5, 6, 7]`.

* Constraints:
  * 1 <= matrix.length, matrix[i].length <= 100
  * 0 <= matrix[i][j] <= 2^31 - 1

### Core Logic: Top-Down DFS with Memoization (The Directed Acyclic Graph)
If we run a standard Depth-First Search (DFS) from every single cell to find the longest path, we will re-evaluate the exact same cells thousands of times, resulting in a Time Limit Exceeded (TLE) error. 

To optimize this to $O(m \cdot n)$, we use **Memoization**. 
We create a cache (a Hash Map or 2D array) where `dp[(r, c)]` permanently stores the length of the longest possible path starting from that exact cell. If another DFS path wanders into a cell we have already solved, we instantly return the cached answer instead of recalculating it.

**The "Strictly Increasing" Secret:**
In standard grid pathfinding algorithms, we normally have to maintain a `visited` set to stop the DFS from walking in infinite circles (e.g., Up -> Right -> Down -> Left). 
Because the path must be *strictly increasing*, it is mathematically impossible to walk in a circle! (You cannot climb up a hill and magically end up at the exact same elevation you started). Therefore, we completely drop the `visited` set, making the code surprisingly elegant.

### Approach: Cached Recursion
1. Initialize a `dp` Hash Map to cache our results. Keys will be `(r, c)` tuples, and values will be the max path length.
2. Define a recursive `dfs(r, c)` function:
   * If `(r, c)` is already in `dp`, return `dp[(r, c)]`.
   * Initialize `res = 1` (the minimum path length is 1, containing just the cell itself).
   * Check all 4 neighbors. If a neighbor is in bounds AND strictly greater than the current cell, recursively call `dfs(nr, nc)`.
   * Update `res = max(res, 1 + dfs(nr, nc))`.
   * Save `res` to `dp[(r, c)]` and return it.
3. Because the absolute longest path could start anywhere, we must loop through every single cell in the matrix and call `dfs(r, c)`.
4. Return the global maximum found.

* **Time Complexity:** $O(m \cdot n)$ — Every cell's maximum path is calculated exactly once and then cached.
* **Space Complexity:** $O(m \cdot n)$ — The `dp` map and the recursive call stack will scale directly with the number of cells in the matrix.

In [19]:
from typing import List

class Solution:
    def longestIncreasingPath(self, matrix: List[List[int]]) -> int:
        ROWS, COLS = len(matrix), len(matrix[0])
        
        # Cache to store the longest path starting from a specific (r, c)
        dp = {} 
        
        def dfs(r, c):
            # Base Case / Memoization Check: 
            # If we've already calculated the answer for this room, return it instantly.
            if (r, c) in dp:
                return dp[(r, c)]
                
            # The absolute minimum path length is 1 (just the current cell)
            res = 1
            
            # Check all 4 adjacent directions
            directions = [[0, 1], [0, -1], [1, 0], [-1, 0]]
            for dr, dc in directions:
                nr, nc = r + dr, c + dc
                
                # Verify the neighbor is in bounds AND strictly strictly greater
                if (0 <= nr < ROWS and 
                    0 <= nc < COLS and 
                    matrix[nr][nc] > matrix[r][c]):
                    
                    # Recursively find the longest path starting from that neighbor,
                    # add 1 for our current step, and keep the maximum result.
                    res = max(res, 1 + dfs(nr, nc))
                    
            # Cache the result before returning so we never have to compute this cell again
            dp[(r, c)] = res
            return res
            
        # We must attempt to start a path from every single cell on the board,
        # tracking the absolute longest one we find.
        longest_path = 0
        for r in range(ROWS):
            for c in range(COLS):
                longest_path = max(longest_path, dfs(r, c))
                
        return longest_path


        """
        =========================================================
        SIMULATION NOTES: matrix = [[1, 2], [3, 4]]
        =========================================================
        dp = {}
        
        Loop starts at (0, 0): matrix[0][0] = 1
          dfs(0, 0):
            Neighbors > 1: (0, 1) [val 2] and (1, 0) [val 3]
            
            Path A -> dfs(0, 1): matrix[0][1] = 2
              Neighbors > 2: (1, 1) [val 4]
              Path A.1 -> dfs(1, 1): matrix[1][1] = 4
                Neighbors > 4: None.
                dp[(1, 1)] = 1
                return 1
              res for (0, 1) = max(1, 1 + 1) = 2
              dp[(0, 1)] = 2
              return 2
              
            Path B -> dfs(1, 0): matrix[1][0] = 3
              Neighbors > 3: (1, 1) [val 4]
              Path B.1 -> dfs(1, 1):
                (1, 1) is already in dp! INSTANT RETURN 1.
              res for (1, 0) = max(1, 1 + 1) = 2
              dp[(1, 0)] = 2
              return 2
              
            res for (0, 0) = max(1, 1+2, 1+2) = 3
            dp[(0, 0)] = 3
            
        Subsequent loop iterations (0,1), (1,0), (1,1) will trigger instant 
        cache returns because they were already solved during the first deep dive!
        
        Final Answer: max(dp.values()) -> 3. (Path: 1 -> 2 -> 4 or 1 -> 3 -> 4)
        """

### Problem 009: Distinct Subsequences (LeetCode 115) [HARD]

### Problem Definition and Constraints
You are given two strings `s` and `t`. Return the number of distinct subsequences of `s` that perfectly match `t`.
A subsequence is derived by deleting some or no characters from a string without changing the relative order of the remaining characters.

**Examples:**
* **Example 1:**
  * **Input:** `s = "caaat", t = "cat"`
  * **Output:** `3`
  * **Explanation:** You can form "cat" by keeping the 'c', the 't', and choosing any ONE of the three 'a's.
* **Example 2:**
  * **Input:** `s = "xxyxy", t = "xy"`
  * **Output:** `5`

* Constraints:
  * 1 <= s.length, t.length <= 1000
  * `s` and `t` consist of English letters.

### Core Logic: The "Include or Skip" Decision Tree
This is a classic string alignment problem, but with a counting twist. When evaluating `s[i]` against `t[j]`, we face two realities:

1. **The Skip (Always available):** Even if `s[i]` perfectly matches `t[j]`, we are never *forced* to use it. We might want to skip it and use a different matching character later in the string. So, we always carry forward the combinations from `dfs(i + 1, j)`.
2. **The Include (Conditional):** If the characters match (`s[i] == t[j]`), we unlock a *second* path. We can consume both characters and move on to match the rest of the strings: `dfs(i + 1, j + 1)`.

Because we want the *total* number of distinct subsequences, we simply add the results of both paths together! 

**The Base Cases:**
*   If we successfully reach the end of `t`, it means we matched every character. We return `1` (we found one valid combination).
*   If we run out of characters in `s` but `t` still has characters left, it's a failed path. We return `0`.

### Approach: Top-Down DFS with Memoization
1. Initialize a `cache` (Hash Map) to store the results of `(i, j)` states.
2. Define a recursive `dfs(i, j)` function:
   * **Base Case 1:** `j == len(t)` -> Return 1.
   * **Base Case 2:** `i == len(s)` -> Return 0.
   * **Memo Check:** If `(i, j)` is in `cache`, return it.
3. Calculate the branches:
   * Always calculate the "skip" path: `res = dfs(i + 1, j)`.
   * If `s[i] == t[j]`, add the "include" path: `res += dfs(i + 1, j + 1)`.
4. Cache `res` at `(i, j)` and return it.
5. Kick off the recursion from `dfs(0, 0)`.

* **Time Complexity:** $O(m \cdot n)$ — Where $m$ is the length of `s` and $n$ is the length of `t`. Because of memoization, we compute each `(i, j)` pair exactly once.
* **Space Complexity:** $O(m \cdot n)$ — The recursive call stack and the hash map can grow up to the number of character combinations between the two strings.

In [20]:
class Solution:
    def numDistinct(self, s: str, t: str) -> int:
        cache = {}
        
        # i points to the current character in s
        # j points to the current character in t
        def dfs(i, j):
            
            # Base Case 1: We successfully matched every character in t!
            if j == len(t):
                return 1
                
            # Base Case 2: We ran out of characters in s, but t is unfinished.
            if i == len(s):
                return 0
                
            # Check Cache: Avoid duplicate overlapping calculations
            if (i, j) in cache:
                return cache[(i, j)]
                
            # Choice 1: We ALWAYS have the option to skip the current character in s.
            # We advance i, but leave j where it is to try matching it later.
            res = dfs(i + 1, j)
            
            # Choice 2: If the characters match, we unlock the option to include it.
            if s[i] == t[j]:
                # We advance BOTH pointers and add these new paths to our total.
                res += dfs(i + 1, j + 1)
                
            # Save the total combinations for this state and return
            cache[(i, j)] = res
            return res
            
        return dfs(0, 0)


        """
        =========================================================
        SIMULATION NOTES: s = "caaat", t = "cat"
        =========================================================
        
        dfs(0, 0) -> 'c' vs 'c'. Match! 
          Path A (Include): dfs(1, 1) -> 'a' vs 'a'. 
          Path B (Skip): dfs(1, 0) -> 'a' vs 'c'.
          
        Let's follow Path A: dfs(1, 1) -> 'a' vs 'a'. Match!
          Path A.1 (Include): dfs(2, 2) -> 'a' vs 't'. (Mismatch -> Skip)
          Path A.2 (Skip): dfs(2, 1) -> 'a' vs 'a'. Match!
          
        Notice what happens at Path A.2: we skipped the first 'a' in s, 
        but we are still matching the first 'a' in t. This effectively 
        evaluates using the SECOND 'a' in s! 
        
        Eventually, these branches hit the end of t (returning 1s) or the 
        end of s (returning 0s). The 1s bubble all the way back up to the 
        root, adding together. 
        
        Since there are 3 different 'a's that can successfully branch into a 
        complete "cat", the root dfs(0, 0) receives 1 + 1 + 1 = 3.
        """

### Problem 010: Burst Balloons (LeetCode 312) [HARD]

### Problem Definition and Constraints
You are given an array of integers `nums` representing balloons. If you burst the `i`-th balloon, you receive `nums[i - 1] * nums[i] * nums[i + 1]` coins. 
If `i - 1` or `i + 1` goes out of bounds, assume the out-of-bounds value is `1`.
Return the maximum number of coins you can receive by bursting all the balloons.

**Examples:**
* **Example 1:**
  * **Input:** `nums = [4,2,3,7]`
  * **Output:** `143`
  * **Explanation:**
    Burst 2: `[4,2,3,7]` -> `4*2*3 = 24`. Remaining: `[4,3,7]`
    Burst 3: `[4,3,7]` -> `4*3*7 = 84`. Remaining: `[4,7]`
    Burst 4: `[4,7]` -> `1*4*7 = 28`. Remaining: `[7]`
    Burst 7: `[7]` -> `1*7*1 = 7`. Remaining: `[]`
    Total: `24 + 84 + 28 + 7 = 143`.

* Constraints:
  * 1 <= nums.length <= 300
  * 0 <= nums[i] <= 100

### Core Logic: The "Reverse Time" Trick (Divide and Conquer)
This is one of the most notoriously difficult DP problems because it breaks the cardinal rule of Dynamic Programming: **Optimal Substructure**.
If you burst a balloon, the balloons to its left and right suddenly become adjacent. The left subproblem and the right subproblem are now permanently tangled together. You cannot solve them independently.

To fix this, we must think in **reverse**. 
Instead of asking, *"Which balloon should I burst first?"*, we ask, *"Which balloon should I burst **LAST**?"*

If we decide that balloon `i` will be the absolute *last* balloon to burst in the subarray from index `l` to `r`, a beautiful mathematical isolation happens:
1. Because balloon `i` survives until the very end, it acts as a solid, unbreakable wall between the left subarray `(l, i - 1)` and the right subarray `(i + 1, r)`. 
2. The left and right subarrays can now be solved completely independently!
3. When it is finally time for balloon `i` to burst, all other balloons in the `(l, r)` range are already gone. Its only surviving neighbors are the balloons sitting strictly outside the bounds of our current subproblem: `nums[l - 1]` and `nums[r + 1]`.

### Approach: Top-Down DFS with Memoization
1. **Pad the Array:** Add a `1` to the beginning and end of the `nums` array. This completely eliminates the need for messy out-of-bounds `if` statements.
2. **Initialize Cache:** Create a memoization hash map or 2D array to store the max coins for any range `(l, r)`.
3. **The DFS Function:** `dfs(l, r)` calculates the max coins obtainable from bursting all balloons strictly between boundaries `l` and `r`.
   * **Base Case:** If `l > r`, there are no balloons left to burst in this range. Return `0`.
   * **The Sweep:** Iterate index `i` from `l` to `r`. Treat every single balloon as if it were the *last* one to burst.
   * **The Calculation:** The coins gained by bursting `i` last is `nums[l - 1] * nums[i] * nums[r + 1]`. Add this to the recursively calculated maximums of the independent left half `dfs(l, i - 1)` and right half `dfs(i + 1, r)`.
   * **Update Max:** Keep track of the highest combination found.
4. Call `dfs(1, len(nums) - 2)` to evaluate the original un-padded balloons.

* **Time Complexity:** $O(n^3)$ — There are $O(n^2)$ possible `(l, r)` subproblems. For each subproblem, we run a loop of size up to $n$ to test every balloon as the "last" one.
* **Space Complexity:** $O(n^2)$ — The cache stores results for all $O(n^2)$ possible combinations of `l` and `r`.

In [21]:
from typing import List

class Solution:
    def maxCoins(self, nums: List[int]) -> int:
        # Pad the array with 1s to handle edge boundaries effortlessly
        nums = [1] + nums + [1]
        cache = {}
        
        # dfs(l, r) returns the max coins we can get by bursting balloons 
        # strictly in the index range [l, r] inclusive.
        def dfs(l, r):
            # Base Case: No balloons left in this range
            if l > r:
                return 0
                
            # Return cached result to prevent redundant O(n^3) branching
            if (l, r) in cache:
                return cache[(l, r)]
                
            # Track the maximum coins we can achieve for this specific (l, r) range
            max_coins = 0
            
            # Iterate i through every balloon in the current range.
            # We are evaluating the reality where balloon 'i' is the LAST 
            # balloon to burst in this specific range.
            for i in range(l, r + 1):
                
                # Because 'i' is the last to burst in the range [l, r], 
                # every other balloon between l and r is already gone. 
                # Therefore, its direct neighbors MUST be l-1 and r+1!
                coins = nums[l - 1] * nums[i] * nums[r + 1]
                
                # Add the maximum coins we can get from the left and right subarrays.
                # Because 'i' acts as a wall until the very end, these halves never interact.
                total_coins = coins + dfs(l, i - 1) + dfs(i + 1, r)
                
                max_coins = max(max_coins, total_coins)
                
            cache[(l, r)] = max_coins
            return max_coins
            
        # Call DFS on the original bounds of the array (ignoring the padded 1s)
        return dfs(1, len(nums) - 2)


        """
        =========================================================
        SIMULATION NOTES: nums = [3, 1, 5]
        =========================================================
        Padded nums: [1, 3, 1, 5, 1]
        Evaluating dfs(1, 3): Range covers [3, 1, 5]
        
        Assume i = 2 (The balloon with value 1). 
        We decide '1' will burst LAST.
        
        Left subproblem: dfs(1, 1) -> Bursting only '3'
          i = 1. Neighbors when it bursts last are l-1 (index 0) and r+1 (index 2).
          Coins = nums[0] * nums[1] * nums[2] = 1 * 3 * 1 = 3.
          Left result = 3.
          
        Right subproblem: dfs(3, 3) -> Bursting only '5'
          i = 3. Neighbors when it bursts last are l-1 (index 2) and r+1 (index 4).
          Coins = nums[2] * nums[3] * nums[4] = 1 * 5 * 1 = 5.
          Right result = 5.
          
        Now, burst '1' (which we saved for last in dfs(1,3)):
          Neighbors are l-1 (index 0) and r+1 (index 4).
          Coins = nums[0] * nums[2] * nums[4] = 1 * 1 * 1 = 1.
          
        Total for choosing '1' last = 1 + 3 + 5 = 9.
        
        The loop will also test choosing '3' last, and choosing '5' last. 
        It finds choosing '5' last yields the true maximum (15 + 0 + 5 = 20), 
        caches it, and returns.
        """

### Problem 011: Regular Expression Matching (LeetCode 10) [HARD]

### Problem Definition and Constraints
You are given an input string `s` and a pattern `p`. Return `true` if the pattern completely matches the input string.
*   `.` Matches any single character.
*   `*` Matches zero or more of the *preceding* element.

**Examples:**
* **Example 1:**
  * **Input:** `s = "aa", p = ".b"`
  * **Output:** `false`
  * **Explanation:** The `.` matches the first 'a', but 'b' cannot match the second 'a'.
* **Example 2:**
  * **Input:** `s = "nnn", p = "n*"`
  * **Output:** `true`
  * **Explanation:** `*` allows the preceding element 'n' to repeat zero or more times. It repeats 3 times to match "nnn".
* **Example 3:**
  * **Input:** `s = "xyz", p = ".*z"`
  * **Output:** `true`
  * **Explanation:** `.*` means "zero or more of any character". It consumes "xy", and the final 'z' matches perfectly.

* Constraints:
  * 1 <= s.length <= 20
  * 1 <= p.length <= 20
  * Each appearance of `*` will be preceded by a valid character or `.`.

### Core Logic: The Star Branching Decision Tree
Standard string matching compares `s[i]` against `p[j]` and increments both pointers. The wildcard `.` is trivial because it acts exactly like a match. The true complexity comes from the `*` modifier.

Because `*` modifies the character *immediately preceding it*, we must always look ahead at `p[j + 1]`. If the next character in the pattern is a `*`, we face a crucial, reality-splitting choice:

1. **The Skip (Zero Occurrences):** We decide the `*` means the preceding character appears *zero* times. We delete the character and the `*` from our pattern. Physically, we advance `j` by 2, and leave `i` right where it is.
2. **The Consume (One or More Occurrences):** If the current characters actually match (`s[i] == p[j]` or `p[j] == '.'`), we can use the `*` to consume `s[i]`. We advance `i` by 1 to move to the next character in the string, but we *keep `j` exactly where it is*. This allows the `*` to be used again and again on the next loop, matching infinitely.

If either of these branching paths eventually reaches the end of both strings successfully, the entire pattern is a match.

### Approach: Top-Down DFS with Memoization
1. Initialize a `cache` hash map to store computed `(i, j)` boolean results.
2. Define `dfs(i, j)` where `i` is the index of `s`, and `j` is the index of `p`:
   * **Base Case:** If `j == len(p)`, we ran out of pattern. This is only a valid match if we also ran out of string (`i == len(s)`).
   * **Memo Check:** If `(i, j)` is in the `cache`, return it.
   * **Match Check:** Evaluate if the current single character matches: `match = i < len(s) and (s[i] == p[j] or p[j] == '.')`.
   * **Star Logic:** If `j + 1 < len(p)` and `p[j + 1] == '*'`:
     We return `True` if *either* the "Skip" path `dfs(i, j + 2)` OR the "Consume" path `(match and dfs(i + 1, j))` is `True`.
   * **Standard Match:** If no star is involved, we just need `match` to be `True` and the rest of the string to match `dfs(i + 1, j + 1)`.
   * Otherwise, return `False`.
3. Cache the result at `(i, j)` before returning.

* **Time Complexity:** $O(m \cdot n)$ — Where $m$ is the length of `s` and $n$ is the length of `p`. Memoization ensures we evaluate every index combination exactly once.
* **Space Complexity:** $O(m \cdot n)$ — The recursive call stack and the dictionary cache will store up to $m \times n$ states.

In [22]:
class Solution:
    def isMatch(self, s: str, p: str) -> bool:
        cache = {}
        
        def dfs(i, j):
            # If we already calculated this specific string/pattern index pairing
            if (i, j) in cache:
                return cache[(i, j)]
                
            # Base Case: If the pattern is exhausted, the string MUST also be exhausted
            if j == len(p):
                return i == len(s)
                
            # 1. Evaluate the current single character
            # We must ensure 'i' is in bounds (j is guaranteed in bounds by the base case)
            match = i < len(s) and (s[i] == p[j] or p[j] == ".")
            
            # 2. Check if a '*' modifier is coming up next
            if j + 1 < len(p) and p[j + 1] == "*":
                
                # Branch A (Skip): Assume zero occurrences. Move j past the char and the '*'.
                # Branch B (Consume): If it matches, consume s[i] by moving i forward, 
                #                     but keep j exactly where it is to use the '*' again.
                cache[(i, j)] = dfs(i, j + 2) or (match and dfs(i + 1, j))
                return cache[(i, j)]
                
            # 3. Standard character matching (no star involved)
            if match:
                cache[(i, j)] = dfs(i + 1, j + 1)
                return cache[(i, j)]
                
            # 4. Characters don't match, and there's no star to save us
            cache[(i, j)] = False
            return False
            
        return dfs(0, 0)


        """
        =========================================================
        SIMULATION NOTES: s = "nnn", p = "n*"
        =========================================================
        
        dfs(0, 0): s[0]='n', p[0]='n'. Match = True. Next is '*'.
          Branch A (Skip): dfs(0, 2)
            j=2 (len of p). i=0 != 3. Returns False.
          Branch B (Consume): match (True) and dfs(1, 0)
          
          Evaluating dfs(1, 0): s[1]='n', p[0]='n'. Match = True. Next is '*'.
            Branch A (Skip): dfs(1, 2) -> False.
            Branch B (Consume): match (True) and dfs(2, 0)
            
            Evaluating dfs(2, 0): s[2]='n', p[0]='n'. Match = True. Next is '*'.
              Branch A (Skip): dfs(2, 2) -> False.
              Branch B (Consume): match (True) and dfs(3, 0)
              
              Evaluating dfs(3, 0): i=3. Match = False (out of bounds). Next is '*'.
                Branch A (Skip): dfs(3, 2)
                  j=2 (len of p). i=3 == len(s). Returns True!
                Branch B (Consume): match (False).
                
              dfs(3, 0) returns True.
            dfs(2, 0) returns True.
          dfs(1, 0) returns True.
        dfs(0, 0) returns True.
        """